In [ ]:
# Imports ----------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cvxpy as cp

sns.set_theme(style="whitegrid")

In [ ]:
# Load Real Electricity Demand Data -----------------------

df = pd.read_csv("AEP_hourly.csv")
df.head()
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.sort_values("Datetime")

df = df.rename(columns={"AEP_MW": "Demand"})
df = df.reset_index(drop=True)

df = df.iloc[:24*7].copy()

In [ ]:
# Simulated Price Data --------------------------------------------------

hours = np.arange(len(df))

price = 30 + 40 * np.where((hours % 24 >= 14) & (hours % 24 <= 20), 1, 0)
price = price + np.random.normal(0, 3, len(df))

df["Price"] = np.clip(price, 5, None)

In [ ]:
# Visualize Real Data, Plot 1 -----------------------------

plt.figure(figsize=(12,4))
sns.lineplot(data=df, x=df.index, y="Demand")
plt.title("AEP Electricity Demand (Real Data)")
plt.show()

# Plot 2 -------------------------------------------------

plt.figure(figsize=(12,4))
sns.lineplot(data=df, x=df.index, y="Price")
plt.title("Simulated Electricity Price Signal")
plt.show()

In [ ]:
# Optimization Model

T = len(df)

charge = cp.Variable(T)
discharge = cp.Variable(T)
battery = cp.Variable(T)
grid = cp.Variable(T)

demand = df["Demand"].values
price = df["Price"].values

capacity = 50000
charge_max = 5000
discharge_max = 5000

objective = cp.Minimize(cp.sum(cp.multiply(price, grid)))

# Energy Balance Constraint ----------------------------------------

constraints = []

for t in range(T):
    constraints += [
        grid[t] + discharge[t] == demand[t] + charge[t],
        charge[t] >= 0,
        discharge[t] >= 0,
        grid[t] >= 0,
        charge[t] <= charge_max,
        discharge[t] <= discharge_max,
        battery[t] >= 0,
        battery[t] <= capacity
    ]

    if t == 0:
        constraints += [battery[t] == charge[t] - discharge[t]]
    else:
        constraints += [
            battery[t] == battery[t-1] + charge[t] - discharge[t]
        ]

# Ramp Constraint --------------------------------------------------

ramp_limit = 3000

for t in range(1, T):
    constraints += [
        cp.abs(grid[t] - grid[t-1]) <= ramp_limit
    ]

# Solution + Results ------------------------------------------------

problem = cp.Problem(objective, constraints)
problem.solve(solver=cp.OSQP)

print("Total cost:", problem.value)

In [ ]:
# Baseline Comparison --------------------------------

baseline_cost = np.sum(price * demand)
optimized_cost = problem.value

print("Baseline cost:", baseline_cost)
print("Optimized cost:", optimized_cost)
print("Savings:", baseline_cost - optimized_cost)

# Peak Shaving ------------------------------------

peak_original = np.max(demand)
peak_new = np.max(grid.value)

print("Peak demand:", peak_original)
print("Peak grid usage:", peak_new)
print("Reduction:", peak_original - peak_new)

# Load Smoothing ----------------------------------

variability_original = np.std(demand)
variability_new = np.std(grid.value)

print("Original variability:", variability_original)
print("New variability:", variability_new)


In [ ]:
# Visualize Optimization --------------------------------------------------

plot_df = pd.DataFrame({
    "Hour": np.arange(T),
    "Demand": demand,
    "Grid": grid.value
})

sns.lineplot(data=plot_df, x="Hour", y="Demand", label="Demand")
sns.lineplot(data=plot_df, x="Hour", y="Grid", label="Optimized Grid")
plt.title("Demand vs Optimized Grid Usage")
plt.show()

# Battery Behavior -----------------------------------------------------

sns.lineplot(x=np.arange(T), y=battery.value)
plt.title("Battery State of Charge")
plt.show()

In [ ]:
# Optimization Function ---------------------------------------------

def run_optimization(capacity):

    T = len(demand)

    charge = cp.Variable(T)
    discharge = cp.Variable(T)
    battery = cp.Variable(T)
    grid = cp.Variable(T)

    charge_max = 5000
    discharge_max = 5000
    ramp_limit = 3000

    objective = cp.Minimize(cp.sum(cp.multiply(price, grid)))

    constraints = []

    for t in range(T):
        constraints += [
            grid[t] + discharge[t] == demand[t] + charge[t],
            charge[t] >= 0,
            discharge[t] >= 0,
            grid[t] >= 0,
            charge[t] <= charge_max,
            discharge[t] <= discharge_max,
            battery[t] >= 0,
            battery[t] <= capacity
        ]

        if t == 0:
            constraints += [battery[t] == charge[t] - discharge[t]]
        else:
            constraints += [
                battery[t] == battery[t-1] + charge[t] - discharge[t]
            ]

    for t in range(1, T):
        constraints += [
            cp.abs(grid[t] - grid[t-1]) <= ramp_limit
        ]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.OSQP)

    # Metrics
    cost = problem.value
    peak = np.max(grid.value)
    variability = np.std(grid.value)

    return cost, peak, variability

baseline_cost = np.sum(price * demand)
baseline_peak = np.max(demand)
baseline_variability = np.std(demand)

results = []

# No Battery ----------------------------------------------
results.append({
    "Case": "No Battery",
    "Cost": baseline_cost,
    "Peak": baseline_peak,
    "Variability": baseline_variability
})

# Small Battery --------------------------------------------
cost, peak, var = run_optimization(capacity=20000)
results.append({
    "Case": "Small Battery",
    "Cost": cost,
    "Peak": peak,
    "Variability": var
})

# Medium Battery --------------------------------------------
cost, peak, var = run_optimization(capacity=50000)
results.append({
    "Case": "Medium Battery",
    "Cost": cost,
    "Peak": peak,
    "Variability": var
})

# Large Battery --------------------------------------------
cost, peak, var = run_optimization(capacity=100000)
results.append({
    "Case": "Large Battery",
    "Cost": cost,
    "Peak": peak,
    "Variability": var
})

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Cost Comparison --------------------------------------------------

sns.barplot(data=results_df, x="Case", y="Cost")
plt.title("Total Cost by Scenario")
plt.show()

# Peak Comparsion ----------------------------------------------

sns.barplot(data=results_df, x="Case", y="Peak")
plt.title("Peak Demand by Scenario")
plt.show()

# Variability Comparison --------------------------------------------

sns.barplot(data=results_df, x="Case", y="Variability")
plt.title("Grid Variability by Scenario")
plt.show()

In [ ]:
# Percentage Improvements --------------------------------------

results_df["Cost Reduction (%)"] = (
    (baseline_cost - results_df["Cost"]) / baseline_cost * 100
)

results_df["Peak Reduction (%)"] = (
    (baseline_peak - results_df["Peak"]) / baseline_peak * 100
)

results_df
plot_df = results_df[results_df["Case"] != "No Battery"]

# Plot 1 -------------------------------------------------------------

plt.figure()
ax = sns.barplot(data=plot_df, x="Case", y="Cost Reduction (%)")

for i, row in plot_df.iterrows():
    ax.text(i, row["Cost Reduction (%)"], f"{row['Cost Reduction (%)']:.1f}%", ha='center')

plt.title("Cost Reduction (%) by Battery Size")
plt.show()

# Plot 2 ----------------------------------------------------------------

plt.figure()
ax = sns.barplot(data=plot_df, x="Case", y="Peak Reduction (%)")

for i, row in plot_df.iterrows():
    ax.text(i, row["Peak Reduction (%)"], f"{row['Peak Reduction (%)']:.1f}%", ha='center')

plt.title("Peak Demand Reduction (%)")
plt.show()

# Plot 3 ----------------------------------------------------------------------------------

plt.figure()
ax = sns.barplot(data=plot_df, x="Case", y="Variability Reduction (%)")

for i, row in plot_df.iterrows():
    ax.text(i, row["Variability Reduction (%)"], f"{row['Variability Reduction (%)']:.1f}%", ha='center')

plt.title("Grid Variability Reduction (%)")
plt.show()

# Combined Figure --------------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Cost Reduction
sns.barplot(data=plot_df, x="Case", y="Cost Reduction (%)", ax=axes[0])
axes[0].set_title("Cost Reduction (%)")

# Peak Reduction
sns.barplot(data=plot_df, x="Case", y="Peak Reduction (%)", ax=axes[1])
axes[1].set_title("Peak Reduction (%)")

# Variability Reduction (if exists)
sns.barplot(data=plot_df, x="Case", y="Variability Reduction (%)", ax=axes[2])
axes[2].set_title("Variability Reduction (%)")

plt.tight_layout()
plt.show()

## Results Interpretation

The results show that adding battery storage consistently reduces total electricity cost, peak demand, and overall grid variability. As battery capacity increases, the system is able to shift more energy away from high-price periods, leading to greater cost savings.

We also observe diminishing returns: while moving from no battery to a small or medium battery yields significant improvements, further increases in capacity provide smaller incremental benefits. This suggests that beyond a certain point, additional storage has limited economic value under the given conditions.

In addition to cost savings, the reduction in peak demand and variability indicates that battery storage can improve grid stability by smoothing fluctuations in demand. Overall, these results highlight the role of storage as a flexible resource for both economic optimization and reliable grid operation.

# Conclusion

This project combines data analysis, machine learning, and optimization to study electricity demand and the role of battery storage in power systems. Using real-world data from the AEP region, the analysis identifies key demand patterns and demonstrates how these patterns can be used for forecasting.

Building on this, the optimization model shows how battery storage can reduce costs, shift energy usage away from peak periods, and improve grid stability. Scenario analysis highlights how these benefits depend on storage capacity and operational constraints, with diminishing returns at higher capacity levels.

Overall, this project illustrates how data-driven methods and optimization can be used together to support more efficient and reliable energy systems.